# 📗 Bài 1 — Vector & ma trận

**Tuần 5** · Toàn bộ bài này là NumPy. Không có chứng minh toán học nào — chỉ code và nhìn kết quả.

In [ ]:
import numpy as np

rng = np.random.default_rng(42)   # seed để chạy lại ra kết quả giống hệt

## 1. Tạo mảng

In [ ]:
v = np.array([1, 2, 3])                  # vector (1 chiều)
M = np.array([[1, 2, 3], [4, 5, 6]])     # ma trận (2 chiều)

for ten, a in [("v", v), ("M", M)]:
    print(f"{ten}: shape={a.shape}  ndim={a.ndim}  size={a.size}  dtype={a.dtype}")
print()
print(M)

> 🔑 `.shape` là thứ bạn sẽ in ra **nhiều nhất** khi debug code AI. 90% lỗi là lỗi shape.

In [ ]:
print("zeros:\n", np.zeros((2, 3)))
print("ones :", np.ones(3))
print("arange:", np.arange(0, 10, 2))
print("linspace:", np.linspace(0, 1, 5))
print("random:", rng.normal(0, 1, size=4).round(3))

## 2. Cắt lát & lập chỉ mục

In [ ]:
M = np.arange(12).reshape(3, 4)
print(M, "\n")

print("M[0]      =", M[0])          # hàng đầu
print("M[:, 0]   =", M[:, 0])       # TẤT CẢ hàng, cột 0
print("M[1, 2]   =", M[1, 2])       # hàng 1 cột 2
print("M[1:, :2] =\n", M[1:, :2])

In [ ]:
# Lập chỉ mục theo điều kiện — dùng rất nhiều
print("M > 5      :\n", M > 5)
print("\nM[M > 5]   =", M[M > 5])
print("np.where   :\n", np.where(M > 5, 1, 0))

## 3. ⚠️ `a * b` KHÁC `a @ b`

In [ ]:
a = np.array([1, 2, 3])
b = np.array([4, 5, 6])

print("a * b =", a * b, "  <- nhân TỪNG PHẦN TỬ, kết quả là MẢNG")
print("a @ b =", a @ b, "         <- TÍCH VÔ HƯỚNG, kết quả là MỘT SỐ")
print()
print("kiểm chứng: 1*4 + 2*5 + 3*6 =", 1*4 + 2*5 + 3*6)

> Nhầm hai phép này là bug kinh điển, và nó **không báo lỗi** — chỉ cho kết quả sai. Cách phát hiện: kiểm tra `.shape` của kết quả.

## 4. Tích vô hướng đo cái gì?

In [ ]:
cac_cap = [
    ("cùng hướng     ", [1, 0], [1, 0]),
    ("gần cùng hướng ", [1, 0], [0.9, 0.4]),
    ("vuông góc      ", [1, 0], [0, 1]),
    ("ngược hướng    ", [1, 0], [-1, 0]),
]

for mo_ta, x, y in cac_cap:
    print(f"{mo_ta} {str(x):<10} · {str(y):<12} = {np.array(x) @ np.array(y):6.2f}")

```
  Cùng hướng          Vuông góc            Ngược hướng
      ↗ ↗                 ↑ →                  ↗ ↙
  tích LỚN, dương      tích ≈ 0           tích ÂM
```

Nhưng tích vô hướng bị ảnh hưởng bởi **độ lớn** của vector:

In [ ]:
x = np.array([1, 0])
print("[1,0] · [1, 0]    =", x @ np.array([1, 0]))
print("[1,0] · [100, 0]  =", x @ np.array([100, 0]), "  <- cùng hướng y hệt, số khác hẳn!")

## 5. Cosine similarity — loại bỏ ảnh hưởng độ lớn

$$\cos(a,b) = \frac{a \cdot b}{|a| \times |b|}$$

In [ ]:
def cosine(a, b):
    a, b = np.asarray(a, float), np.asarray(b, float)
    na, nb = np.linalg.norm(a), np.linalg.norm(b)
    if na == 0 or nb == 0:
        return 0.0
    return float(a @ b / (na * nb))


for mo_ta, x, y in cac_cap:
    print(f"{mo_ta} cosine = {cosine(x, y):6.3f}")

print()
print("[1,0] vs [100,0]  cosine =", cosine([1, 0], [100, 0]), " <- độ lớn không còn ảnh hưởng")

| Giá trị | Nghĩa |
|---|---|
| `1.0` | Cùng hướng hoàn toàn — rất giống nhau |
| `0.0` | Không liên quan |
| `-1.0` | Ngược hướng hoàn toàn |

> 🔗 **Đây là công thức trái tim của Phase 08 (RAG).** Đoạn văn dài có vector "lớn" hơn đoạn ngắn; nếu dùng tích vô hướng thô thì đoạn dài luôn thắng. Cosine chuẩn hoá điều đó.

## 6. Mô phỏng tìm kiếm ngữ nghĩa

Giả sử mỗi câu đã được biến thành vector 4 chiều (thật ra là 1536 chiều, nhưng nguyên lý y hệt).

In [ ]:
kho_van_ban = {
    "Mèo là động vật nuôi trong nhà": np.array([0.9, 0.1, 0.0, 0.2]),
    "Chó rất trung thành với chủ":    np.array([0.85, 0.15, 0.05, 0.1]),
    "Python là ngôn ngữ lập trình":   np.array([0.0, 0.1, 0.95, 0.3]),
    "Java cũng dùng để lập trình":    np.array([0.05, 0.0, 0.9, 0.35]),
}

cau_hoi = np.array([0.0, 0.05, 0.92, 0.3])      # ~ "học lập trình thế nào"

diem = {cau: cosine(cau_hoi, vec) for cau, vec in kho_van_ban.items()}

print("Câu hỏi liên quan nhất tới:\n")
for cau, d in sorted(diem.items(), key=lambda x: -x[1]):
    thanh = "█" * int(d * 30)
    print(f"  {d:5.3f} {thanh:<30} {cau}")

Hai câu về lập trình xếp đầu, hai câu về động vật xếp cuối — **dù không có một từ nào trùng nhau**. Đó chính là *tìm kiếm ngữ nghĩa*.

## 7. Nhân ma trận — phép tính lõi của AI

In [ ]:
A = np.array([[1, 2], [3, 4], [5, 6]])     # (3, 2)
B = np.array([[1, 0, 1, 0], [0, 1, 0, 1]])  # (2, 4)

print("A.shape =", A.shape)
print("B.shape =", B.shape)
print("A @ B shape =", (A @ B).shape, "  <- (3,2) @ (2,4) -> (3,4)")
print()
print(A @ B)

In [ ]:
# @loi-co-y  <- ô này CỐ Ý lỗi, dùng để bạn xem thông báo lỗi
# Shape không khớp -> lỗi ngay. Chạy để thấy thông báo lỗi trông thế nào.
A @ A

```
   (m, n)  @  (n, p)   →   (m, p)
       ↑       ↑
       └───────┘  PHẢI bằng nhau
```

Đọc thông báo lỗi: `matmul: Input operand 1 has a mismatch...` — NumPy nói rõ hai chiều nào không khớp. Học đọc nó ngay từ bây giờ.

## 8. Một lớp mạng neural

In [ ]:
# X: 5 mẫu, mỗi mẫu 3 đặc trưng
X = rng.normal(size=(5, 3))
W = rng.normal(size=(3, 2))        # 3 đầu vào -> 2 đầu ra
b = np.array([0.5, -0.5])

z = X @ W + b                      # phép tuyến tính
out = np.maximum(0, z)             # ReLU — hàm kích hoạt

print("X    ", X.shape)
print("W    ", W.shape)
print("X@W+b", z.shape)
print()
print("Trước ReLU:\n", z.round(2))
print("\nSau ReLU (số âm bị đưa về 0):\n", out.round(2))

```python
output = activation(X @ W + b)
```

Đó là **toàn bộ** một lớp mạng neural. Hồi quy tuyến tính = phép này không có activation. GPT = rất nhiều phép này xen kẽ attention.

Bạn vừa viết xong lõi của deep learning trong 2 dòng.

## 9. Broadcasting

In [ ]:
A = np.array([[1, 2, 3], [4, 5, 6]])     # (2, 3)
v = np.array([10, 20, 30])               # (3,)

print("A + v =\n", A + v)     # v được lặp cho từng hàng
print()
print("A + 100 =\n", A + 100)  # số vô hướng cộng vào mọi phần tử

**Quy tắc:** so shape từ **phải sang trái**; hai chiều tương thích khi bằng nhau hoặc một trong hai bằng 1.

```
(2, 3)  +  (3,)     → (3,) hiểu là (1,3) → OK, ra (2,3)
(3, 1)  +  (1, 4)   → OK, ra (3, 4)
(2, 3)  +  (2,)     → ❌ 3 vs 2 không tương thích
```

In [ ]:
# @loi-co-y  <- ô này CỐ Ý lỗi, dùng để bạn xem thông báo lỗi
# Thử trường hợp sai
A + np.array([10, 20])

## 10. axis=0 hay axis=1?

Đây là chỗ người mới nhầm nhiều nhất.

In [ ]:
M = np.array([[1., 2., 3.],
              [4., 5., 6.]])

print("M =\n", M, "\n")
print("M.sum()        =", M.sum(), "  <- tất cả")
print("M.sum(axis=0)  =", M.sum(axis=0), " <- gộp các HÀNG -> còn lại 3 CỘT")
print("M.sum(axis=1)  =", M.sum(axis=1), "     <- gộp các CỘT -> còn lại 2 HÀNG")

**Mẹo nhớ:** `axis=0` là chiều đi **xuống** ↓ (dọc theo cột). Kết quả có độ dài = số cột.

Trong ML, dữ liệu là `(n_mẫu, n_đặc_trưng)` nên **`axis=0` gần như luôn là cái bạn muốn** — nó tính thống kê cho từng đặc trưng.

## 11. Chuẩn hoá (z-score)

In [ ]:
X = np.array([[25., 15_000_000.],
              [31.,  8_000_000.],
              [45., 30_000_000.],
              [28., 12_000_000.]])
# cột 0 = tuổi, cột 1 = thu nhập

print("Trước chuẩn hoá — hai cột lệch thang đo hàng triệu lần:")
print(X)

Z = (X - X.mean(axis=0)) / X.std(axis=0)
print("\nSau chuẩn hoá:")
print(Z.round(3))
print("\nmean từng cột:", Z.mean(axis=0).round(10))
print("std  từng cột:", Z.std(axis=0).round(10))

Toàn bộ việc chuẩn hoá gói trong **một dòng**, nhờ broadcasting. Không cần vòng lặp nào.

> ⚠️ **Nhớ cho Phase 04:** trong thực tế phải tính `mean`/`std` **chỉ từ tập train**, rồi áp dụng cho test. Tính trên toàn bộ dữ liệu là **data leakage**.

## 12. Vì sao NumPy nhanh hơn vòng lặp — đo thử

In [ ]:
import time

n = 1_000_000
ds_python = list(range(n))
ds_numpy = np.arange(n)

t = time.perf_counter()
kq1 = [x * 2 for x in ds_python]
t_py = time.perf_counter() - t

t = time.perf_counter()
kq2 = ds_numpy * 2
t_np = time.perf_counter() - t

print(f"Python list : {t_py*1000:8.2f} ms")
print(f"NumPy array : {t_np*1000:8.2f} ms")
print(f"NumPy nhanh hơn khoảng {t_py/max(t_np, 1e-9):.0f} lần")

**Ba lý do:**
1. Lõi viết bằng **C đã biên dịch**, không phải Python thông dịch
2. Bộ nhớ **liền mạch, cùng kiểu** → CPU cache hoạt động hiệu quả
3. Tận dụng **SIMD** — CPU xử lý nhiều số bằng một lệnh

> 🔑 **Nguyên tắc:** nếu bạn đang viết `for` để duyệt một mảng NumPy, gần như chắc chắn có cách vectorized nhanh hơn 10–100 lần.

## ✍️ Bài tập

```bash
# Làm exercises/ex01_vector_ma_tran.py rồi chấm
pytest tests/phase02/test_ex01.py -v
```